# Traditional Methods: Basic CNN

## Modeling Approach

Our baseline CNN uses a simple three-layer architecture that serves as a foundational benchmark for diabetic retinopathy classification. The model progressively extracts features through three convolutional blocks (32, 64, and 128 channels) with 3x3 kernels, ReLU activation, and max pooling, reducing input images from 224x224 to 28x28 while learning increasingly complex patterns from basic edges to pathological features. The extracted features are then flattened and passed through two fully connected layers (256 neurons, then 5 output classes) with dropout regularization to prevent overfitting. This straightforward design allows us to establish baseline performance and evaluate whether the added complexity of modern architectures like ResNet and EfficientNet provides meaningful improvements for retinal image classification, while maintaining transparency and interpretability.

In [1]:
import pandas as pd
import numpy as np
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0
from torch.utils.data import DataLoader, Subset, TensorDataset
import torch.nn.functional as F
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score
import matplotlib.pyplot as plt

In [ ]:
x_train = torch.load('/data/x_train.pt')
y_train = torch.load('/data/y_train.pt')
x_test = torch.load('/data/x_test.pt')
y_test = torch.load('/data/y_test.pt')

In [3]:
def qwk(y_true, y_pred, num_classes=5):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

In [4]:
class CNN(nn.Module):
    def __init__(self, num_classes=5):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        # After 3 pooling: 224 -> 112 -> 56 -> 28
        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, num_classes)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

### Model Training

The following function implements the standard training loop for our CNN model. It processes batches of retinal images through forward propagation, calculates loss using our chosen criterion, performs backpropagation to compute gradients, and updates model parameters using the optimizer. The function returns the average training loss across all batches.

In [5]:
def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

### Model Evaluation

This function evaluates our trained model on the test dataset by making predictions on retinal images and collecting both the true labels and predicted severity levels. It returns arrays of actual and predicted classifications that we'll use to calculate performance metrics like accuracy, confusion matrix, and other evaluation measures.

In [6]:
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.array(all_labels), np.array(all_preds)

## Putting It All Together

This main function orchestrates the entire model training and evaluation process by setting up data loaders, initializing our CNN model with loss function and optimizer, training for 20 epochs, and finally evaluating performance using accuracy and quadratic weighted kappa metrics along with class distribution analysis.

In [ ]:
def main(x_train, y_train, x_test, y_test):
    print(f"Train set: {len(x_train)} images")
    print(f"Test set: {len(x_test)} images")
    
    train_dataset = TensorDataset(x_train, y_train)
    test_dataset = TensorDataset(x_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = CNN(num_classes=5).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    print(f"Using device: {device}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Training loop
    epochs = 20
    for epoch in range(epochs):
        train_loss = train_model(model, train_loader, criterion, optimizer, device)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {train_loss:.4f}")
    
    y_true, y_pred = evaluate_model(model, test_loader, device)
    
    kappa = qwk(y_true, y_pred)
    accuracy = (y_true == y_pred).mean()
    
    print(f"\nFinal Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Quadratic Weighted Kappa: {kappa:.4f}")
    
    # Class distribution
    print(f"\nTrue labels distribution: {np.bincount(y_true)}")
    print(f"Predicted labels distribution: {np.bincount(y_pred)}")
    
    return model, kappa

In [10]:
model, kappa = main(x_train, y_train, x_test, y_test)

Train set: 2929 images
Test set: 733 images
Using device: cuda
Model parameters: 25,784,901
Epoch 5/20, Loss: 0.7252
Epoch 10/20, Loss: 0.5062
Epoch 15/20, Loss: 0.2470
Epoch 20/20, Loss: 0.1142

Final Results:
Accuracy: 0.7367
Quadratic Weighted Kappa: 0.7416

True labels distribution: [361  74 200  39  59]
Predicted labels distribution: [372  49 249  23  40]
